Groundwater | Case Study

# Topic 5 : Model Validation

Dr. Xiang-Zhao Kong & Dr. Beatrice Marti & Louise Noel du Payrat

**Step 6 of the Groundwater Modelling Process**

In this notebook, we assess whether our calibrated model can make reliable predictions beyond the data used for calibration. This is the critical step that determines if the model is fit for its intended purpose.

**Important**: This notebook also demonstrates how to handle a common real-world challenge: **insufficient observation data for proper validation**. Learning to recognize and communicate data limitations is an essential professional skill.

---

### Learning Objectives

By the end of this notebook, you will be able to:

1. Distinguish between model **verification** and **validation**
2. Recognize when observation data is **insufficient for validation**
3. Apply validation approaches appropriate for **steady-state models**
4. Assess **physical plausibility** of model results
5. Communicate data limitations **professionally to clients**
6. Determine what a model **can and cannot** be used for given validation status

---

### Notebook Structure

1. Introduction to Validation (including data limitations)
2. Model Verification (mass balance, boundary conditions)
3. Load Calibrated Model and Data
4. Model Performance Assessment (all available wells)
5. Validation Against Independent Data Types
6. Physical Plausibility Assessment
7. Residual Analysis
8. Validation Summary
9. Discussion: Data Requirements and Professional Communication

In [ ]:
# Import libraries
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import flopy
from flopy.utils import HeadFile, CellBudgetFile

from IPython.display import display

# Load local modules
sys.path.append('../SUPPORT_REPO/src')
sys.path.append('../SUPPORT_REPO/src/scripts/scripts_exercises')

from data_utils import (
    download_named_file,
    get_default_data_folder,
)

# Set up plotting defaults
plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['font.size'] = 12

## 1 - Introduction to Validation

### Verification vs. Validation

These terms are often confused but have distinct meanings:

| Aspect | Verification | Validation |
|--------|--------------|------------|
| **Question** | Is the model solving the equations correctly? | Is the model representing reality well enough? |
| **Focus** | Mathematical correctness | Predictive capability |
| **Methods** | Mass balance, analytical comparisons | Independent data, physical plausibility |
| **When** | Every model run | After calibration |

### The Validation Challenge for Steady-State Models

For **transient models**, validation is straightforward: reserve a portion of the time series data (e.g., 25%) that wasn't used for calibration, then test the model's predictions against this held-out data.

For **steady-state models** like ours, this temporal split isn't possible because:
- We calibrated to long-term **mean** head values
- There's no temporal dimension to split

Instead, we would typically use alternative validation approaches:

1. **Spatial splitting**: Reserve some observation wells for validation
2. **Independent data types**: Test against data not used in calibration (e.g., flux measurements, baseflow estimates)
3. **Physical plausibility**: Verify that model behavior matches our hydrogeological understanding
4. **Cross-validation**: Leave-one-out analysis to assess sensitivity to individual observations

### Critical Limitation: Insufficient Observation Data

> **Reality Check**: Our model has only **3 observation wells**, all located in approximately the same row of the model grid. This is a **severe limitation** for validation:
>
> - **No spatial diversity**: All wells are aligned, so we cannot assess model performance across different areas of the domain
> - **Splitting is meaningless**: A 75/25 split would give us 2 calibration wells and 1 validation well - statistically meaningless
> - **No independent test**: With so few points, any "validation" would have no statistical power
>
> **This is a common real-world situation.** As a professional modeller, this is the point where you would need to have an honest conversation with your client:
>
> *"The available observation data is insufficient for rigorous model validation. With only 3 wells in a single transect, we cannot independently verify model predictions across the domain. To build confidence in this model, we would need additional observation wells distributed across the study area, ideally covering different geological zones and distances from boundaries. Without this data, the model should be considered a screening tool only, not suitable for high-stakes decisions."*

Given this limitation, in this notebook we will:
1. **Focus on verification** (mass balance, boundary conditions) - these don't require independent data
2. **Assess physical plausibility** - does the model behave reasonably?
3. **Use all available wells for assessment** - no artificial split
4. **Be transparent about limitations** - document what we can and cannot conclude

### What Validation Can and Cannot Tell Us

**Validation CAN:**
- Identify gross model errors or misspecifications
- Provide confidence bounds on model predictions
- Support (but not prove) that the model is fit for purpose

**Validation CANNOT:**
- Prove the model is "correct" (all models are simplifications)
- Guarantee accurate predictions under all conditions
- Replace the need for professional judgment
- Overcome fundamental data limitations (like insufficient observation wells)

## 2 - Model Verification

Before assessing predictive capability, we must verify that the model is solving correctly. This is a prerequisite for meaningful validation.

In [ ]:
# Define model paths (same as calibration notebook)
model_name = 'limmat_valley_model_nwt'
workspace = os.path.join(get_default_data_folder(), 'calibration')

# Check if model directory exists
if not os.path.exists(workspace):
    print(f"ERROR: Model workspace not found at: {workspace}")
    print("Please run notebook 5_calibration.ipynb first to create the calibrated model.")
else:
    print(f"Model workspace found at: {workspace}")

In [ ]:
# Load the calibrated MODFLOW-NWT model
mf = flopy.modflow.Modflow.load(
    f"{model_name}.nam",
    model_ws=workspace,
    exe_name='mfnwt',
    version='mfnwt',
    verbose=False
)

print(f"Model loaded: {mf.name}")
print(f"Grid: {mf.nlay} layer(s), {mf.nrow} rows, {mf.ncol} columns")
print(f"Packages: {[pkg.name[0] for pkg in mf.packagelist]}")

# Set grid transformation parameters (from calibration notebook)
grid_rotation_angle = 30  # degrees

### 2.1 - Mass Balance Verification

The most fundamental verification check: does water in = water out?

For a steady-state model, the mass balance error should be essentially zero (< 0.1%).

In [ ]:
# Read water balance from the listing file
list_file = os.path.join(workspace, f"{model_name}.list")

if os.path.exists(list_file):
    # Use FloPy's built-in listing file budget parser
    mfl = flopy.utils.MfListBudget(list_file)
    
    # Get the budget data as a DataFrame
    budget_df = mfl.get_dataframes()[0]
    
    print("="*70)
    print("VERIFICATION: WATER BALANCE")
    print("="*70)
    
    print(f"\n{'COMPONENT':<25} {'IN (m³/day)':>15} {'OUT (m³/day)':>15}")
    print("-"*70)
    
    total_in = 0.0
    total_out = 0.0
    
    # Get the last row (final time step)
    last_row = budget_df.iloc[-1]
    
    # Budget terms come in pairs: TERM_IN and TERM_OUT
    in_cols = [c for c in budget_df.columns if c.endswith('_IN')]
    terms = [c.replace('_IN', '') for c in in_cols]
    
    budget_components = []
    for term in terms:
        in_col = f"{term}_IN"
        out_col = f"{term}_OUT"
        
        inflow = last_row[in_col] if in_col in last_row else 0.0
        outflow = last_row[out_col] if out_col in last_row else 0.0
        
        total_in += inflow
        total_out += outflow
        
        # Only print if there's flow
        if inflow > 0.01 or outflow > 0.01:
            print(f"{term:<25} {inflow:>15.2f} {outflow:>15.2f}")
            budget_components.append({
                'Component': term,
                'Inflow': inflow,
                'Outflow': outflow
            })
    
    print("-"*70)
    print(f"{'TOTAL':<25} {total_in:>15.2f} {total_out:>15.2f}")
    print("="*70)
    
    # Calculate percent discrepancy
    avg_flow = (total_in + total_out) / 2
    if avg_flow > 0:
        discrepancy = (total_in - total_out) / avg_flow * 100
        print(f"\nIN - OUT:            {total_in - total_out:>10.4f} m³/day")
        print(f"Percent Discrepancy: {discrepancy:>10.4f} %")
        
        if abs(discrepancy) < 0.1:
            print("\n✓ VERIFICATION PASSED: Mass balance error < 0.1%")
        elif abs(discrepancy) < 1.0:
            print("\n⚠ Mass balance error < 1% - acceptable but check model")
        else:
            print("\n✗ WARNING: Mass balance error exceeds 1% - investigate!")
    
    # Store for later use
    water_balance = pd.DataFrame(budget_components)
else:
    print(f"Listing file not found: {list_file}")

In [ ]:
# Visualize water balance components
if 'water_balance' in dir() and len(water_balance) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(water_balance))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, water_balance['Inflow'], width, label='Inflow', color='steelblue')
    bars2 = ax.bar(x + width/2, water_balance['Outflow'], width, label='Outflow', color='coral')
    
    ax.set_ylabel('Flow Rate (m³/day)')
    ax.set_title('Figure 1: Water Balance Components')
    ax.set_xticks(x)
    ax.set_xticklabels(water_balance['Component'], rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 2.2 - Boundary Condition Verification

Check that boundary conditions are implemented as intended.

In [ ]:
# Verify boundary conditions
print("="*70)
print("VERIFICATION: BOUNDARY CONDITIONS")
print("="*70)

# Check IBOUND array
if hasattr(mf, 'bas6'):
    ibound = mf.bas6.ibound.array[0]  # Layer 0
    
    n_active = np.sum(ibound > 0)
    n_inactive = np.sum(ibound == 0)
    n_constant = np.sum(ibound < 0)
    
    print(f"\nIBOUND Summary (Layer 1):")
    print(f"  Active cells (IBOUND > 0):    {n_active:,}")
    print(f"  Inactive cells (IBOUND = 0):  {n_inactive:,}")
    print(f"  Constant head (IBOUND < 0):   {n_constant:,}")

# Check for key boundary packages
print(f"\nBoundary Packages:")

if hasattr(mf, 'riv') and mf.riv is not None:
    riv_data = mf.riv.stress_period_data[0]
    print(f"  RIV (River): {len(riv_data)} cells")

if hasattr(mf, 'ghb') and mf.ghb is not None:
    ghb_data = mf.ghb.stress_period_data[0]
    print(f"  GHB (General Head Boundary): {len(ghb_data)} cells")

if hasattr(mf, 'drn') and mf.drn is not None:
    drn_data = mf.drn.stress_period_data[0]
    print(f"  DRN (Drain): {len(drn_data)} cells")

if hasattr(mf, 'chd') and mf.chd is not None:
    chd_data = mf.chd.stress_period_data[0]
    print(f"  CHD (Constant Head): {len(chd_data)} cells")

if hasattr(mf, 'rch') and mf.rch is not None:
    rch_array = mf.rch.rech.array
    if rch_array.ndim == 4:
        rch_2d = rch_array[0, 0]
    elif rch_array.ndim == 3:
        rch_2d = rch_array[0]
    else:
        rch_2d = rch_array
    rch_cells = np.sum(rch_2d > 0)
    rch_mean = np.mean(rch_2d[rch_2d > 0]) * 1000 * 365  # Convert to mm/year
    print(f"  RCH (Recharge): {rch_cells:,} cells, mean = {rch_mean:.1f} mm/year")

## 3 - Load Calibrated Model Results and Observation Data

Now we load the calibration results and prepare the validation dataset.

In [ ]:
# Load simulated heads
head_file = os.path.join(workspace, f"{model_name}.hds")

if os.path.exists(head_file):
    hds = HeadFile(head_file)
    head = hds.get_data(kstpkper=(0, 0))  # Steady state: first time step, first stress period
    print(f"Head array loaded: shape = {head.shape}")
    
    # Mask dry/inactive cells
    head_masked = np.ma.masked_where(head < -999, head)
    print(f"Head range (active cells): {head_masked.min():.2f} to {head_masked.max():.2f} m a.s.l.")
else:
    print(f"Head file not found: {head_file}")
    print("Please run the calibration notebook first.")

In [ ]:
# Load observation data (same as calibration notebook)
gw_ts_path = download_named_file(
    name='groundwater_timeseries',
    data_type='time_series'
)

# Process the data
gw_ts_raw = pd.read_csv(gw_ts_path)
gw_ts = gw_ts_raw[['date', 'value', 'well_id', 'x_coord', 'y_coord']].copy()
gw_ts['date'] = pd.to_datetime(gw_ts['date'], errors='coerce')
gw_ts = gw_ts.drop_duplicates()
gw_ts['well_id'] = gw_ts['well_id'].astype(str)
gw_ts = gw_ts[~gw_ts['well_id'].isin(['nan', 'NaN', ''])]
gw_ts = gw_ts.dropna(subset=['date'])

# Standardize well IDs
gw_ts.loc[gw_ts['well_id'] == 'B53', 'well_id'] = 'B5-3'

print(f"Loaded {len(gw_ts)} observations from {gw_ts['well_id'].nunique()} wells")

# Calculate mean heads for steady-state comparison
mean_heads = gw_ts.groupby('well_id')['value'].mean()

In [ ]:
# Extract unique well locations
well_locations = gw_ts.groupby('well_id').agg({
    'x_coord': 'first',
    'y_coord': 'first'
}).reset_index()

well_locations['x_coord'] = pd.to_numeric(well_locations['x_coord'], errors='coerce')
well_locations['y_coord'] = pd.to_numeric(well_locations['y_coord'], errors='coerce')

print(f"Found {len(well_locations)} unique observation wells")

In [ ]:
# Function to find model cell for a point (from calibration notebook)
def find_cell_for_point(modelgrid, x, y):
    """
    Find the model cell (row, col) containing a point.
    Handles rotated grids by converting to local coordinates first.
    """
    try:
        if modelgrid.angrot != 0:
            local_x, local_y = modelgrid.get_local_coords(x, y)
        else:
            local_x, local_y = x - modelgrid.xoffset, y - modelgrid.yoffset
        
        # Find row (y decreases downward)
        delc = modelgrid.delc
        y_edges = np.concatenate([[0], np.cumsum(delc)])
        y_from_top = modelgrid.nrow * np.mean(delc) - local_y
        
        if y_from_top < 0 or y_from_top > y_edges[-1]:
            return None
        
        row = np.searchsorted(y_edges[1:], y_from_top)
        row = min(row, modelgrid.nrow - 1)
        
        # Find column
        delr = modelgrid.delr
        x_edges = np.concatenate([[0], np.cumsum(delr)])
        
        if local_x < 0 or local_x > x_edges[-1]:
            return None
        
        col = np.searchsorted(x_edges[1:], local_x)
        col = min(col, modelgrid.ncol - 1)
        
        return (row, col)
    except:
        return None

# Match wells to model grid
obs_wells = []

for idx, row in well_locations.iterrows():
    well_id = row['well_id']
    x, y = row['x_coord'], row['y_coord']
    
    if pd.isna(x) or pd.isna(y):
        continue
    
    cell = find_cell_for_point(mf.modelgrid, x, y)
    if cell is not None:
        i, j = cell
        if hasattr(mf, 'bas6') and mf.bas6.ibound.array[0, i, j] > 0:
            obs_wells.append({
                'well_id': well_id,
                'x': x,
                'y': y,
                'row': i,
                'col': j,
                'layer': 0
            })

obs_wells_df = pd.DataFrame(obs_wells)
print(f"\n{len(obs_wells_df)} observation wells matched to active model cells")

### 3.1 - Assessment of Available Observation Data

Let's examine what observation data we have and assess its adequacy for validation.

In [ ]:
# Assess observation data adequacy
n_wells = len(obs_wells_df)

print("="*70)
print("OBSERVATION DATA ASSESSMENT")
print("="*70)

print(f"\nNumber of observation wells: {n_wells}")
print(f"\nWell locations:")
display(obs_wells_df[['well_id', 'x', 'y', 'row', 'col']])

# Check spatial distribution
if n_wells > 1:
    x_range = obs_wells_df['x'].max() - obs_wells_df['x'].min()
    y_range = obs_wells_df['y'].max() - obs_wells_df['y'].min()
    row_range = obs_wells_df['row'].max() - obs_wells_df['row'].min()
    col_range = obs_wells_df['col'].max() - obs_wells_df['col'].min()
    
    print(f"\nSpatial extent of observation wells:")
    print(f"  X range: {x_range:.0f} m")
    print(f"  Y range: {y_range:.0f} m")
    print(f"  Row range: {row_range} cells")
    print(f"  Column range: {col_range} cells")

# Assessment
print("\n" + "="*70)
print("DATA ADEQUACY FOR VALIDATION")
print("="*70)

if n_wells < 5:
    print(f"""
⚠ INSUFFICIENT DATA FOR RIGOROUS VALIDATION

With only {n_wells} observation wells, meaningful validation is NOT possible:

  - Statistical metrics (RMSE, R²) have no significance with n={n_wells}
  - Spatial splitting would leave too few points in each group
  - Cannot assess model performance across different parts of the domain
  - Cannot detect systematic spatial biases

WHAT THIS MEANS FOR THE MODEL:

  This model can be used for:
    ✓ Conceptual understanding of the flow system
    ✓ Screening-level analysis
    ✓ Identifying data gaps
    
  This model should NOT be used for:
    ✗ Regulatory decisions without additional validation
    ✗ Detailed wellhead protection zone delineation
    ✗ Quantitative predictions with high confidence

RECOMMENDATION TO CLIENT:

  "To enable proper model validation, we recommend installing additional
  observation wells distributed across the model domain, particularly:
  - Near model boundaries (to verify boundary conditions)
  - In areas of interest for predictions
  - Across different geological units
  - At varying distances from the river
  
  A minimum of 10-15 wells with good spatial coverage would be needed
  for meaningful validation of a model of this size."
""")
else:
    print(f"\n✓ {n_wells} wells available - may be adequate for basic validation")
    print("  Consider spatial distribution when splitting into calibration/validation sets")

In [ ]:
# Visualize observation well locations and their limited spatial coverage
fig, ax = plt.subplots(figsize=(12, 10))

# Plot model grid with heads
head_masked = np.ma.masked_where(head[0] < -999, head[0])
pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
head_plot = pmv.plot_array(head_masked, cmap='viridis', alpha=0.7)
plt.colorbar(head_plot, ax=ax, label='Simulated Head (m a.s.l.)', shrink=0.5)

# Plot ALL observation wells (no split - we use all for assessment)
ax.scatter(obs_wells_df['x'], obs_wells_df['y'],
           c='red', s=200, marker='o', label=f'Observation wells (n={len(obs_wells_df)})',
           edgecolors='white', linewidth=2, zorder=5)

# Add labels
for _, well in obs_wells_df.iterrows():
    ax.annotate(well['well_id'], (well['x'], well['y']),
                xytext=(5, 5), textcoords='offset points', fontsize=10, 
                fontweight='bold', color='red')

# Add a note about coverage
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Figure 2: Observation Well Locations\n(Note: All wells are in approximately the same row - limited spatial coverage)')
ax.legend(loc='upper left')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\nNote: The limited spatial distribution of wells means we cannot assess")
print("model performance across the full domain. This is a significant limitation.")

## 4 - Model Performance Assessment

Given the data limitations, we assess model performance using **all available wells**. Remember: with only 3 wells, these metrics have limited statistical meaning, but they still provide useful information about model fit in the observed area.

In [ ]:
# Performance metrics function (from calibration notebook)
def calculate_metrics(observed, simulated):
    """
    Calculate calibration/validation performance metrics.
    """
    obs = np.array(observed)
    sim = np.array(simulated)
    
    # Remove NaN values
    mask = ~(np.isnan(obs) | np.isnan(sim))
    obs = obs[mask]
    sim = sim[mask]
    
    n = len(obs)
    if n == 0:
        return None
    
    residuals = sim - obs
    
    # Mean Error (Bias)
    me = np.mean(residuals)
    
    # Mean Absolute Error
    mae = np.mean(np.abs(residuals))
    
    # Root Mean Square Error
    rmse = np.sqrt(np.mean(residuals**2))
    
    # Coefficient of determination (R²)
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((obs - np.mean(obs))**2)
    r2 = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Normalized RMSE
    nrmse = rmse / (np.max(obs) - np.min(obs)) * 100 if (np.max(obs) - np.min(obs)) > 0 else 0
    
    return {
        'N': n,
        'Bias (m)': round(me, 3),
        'MAE (m)': round(mae, 3),
        'RMSE (m)': round(rmse, 3),
        'NRMSE (%)': round(nrmse, 1),
        'R²': round(r2, 3)
    }

In [ ]:
# Extract simulated and observed heads for ALL wells
def add_heads_to_wells(wells_df, head_array, mean_heads):
    """Add simulated and observed heads to wells DataFrame."""
    df = wells_df.copy()
    
    # Extract simulated heads
    sim_heads = []
    for _, well in df.iterrows():
        i, j = int(well['row']), int(well['col'])
        h = head_array[0, i, j]
        sim_heads.append(h if h > -999 else np.nan)
    df['simulated'] = sim_heads
    
    # Add observed heads
    df['observed'] = df['well_id'].map(mean_heads)
    
    # Calculate residuals
    df['residual'] = df['simulated'] - df['observed']
    
    return df

# Add heads to ALL wells (no split)
all_results = add_heads_to_wells(obs_wells_df, head, mean_heads)

print("Model Performance at All Observation Wells:")
print("="*70)
display(all_results[['well_id', 'observed', 'simulated', 'residual']].round(2))

In [ ]:
# Calculate metrics for all wells
all_metrics = calculate_metrics(
    all_results['observed'],
    all_results['simulated']
)

print("="*70)
print("PERFORMANCE METRICS (ALL WELLS)")
print("="*70)

if all_metrics:
    metrics_df = pd.DataFrame([all_metrics]).T
    metrics_df.columns = ['Value']
    display(metrics_df)
    
    print("\n⚠ IMPORTANT CAVEAT:")
    print(f"  These metrics are based on only {all_metrics['N']} observation points.")
    print("  With such limited data:")
    print("  - RMSE and MAE indicate fit at observed locations only")
    print("  - R² has no statistical meaning with n=3")
    print("  - We cannot assess prediction accuracy elsewhere in the domain")
    print("  - Good fit here does NOT guarantee good predictions elsewhere")

In [ ]:
# Simple assessment - no split comparison possible
print("="*70)
print("MODEL FIT ASSESSMENT")
print("="*70)

if all_metrics:
    print(f"\nAt the {all_metrics['N']} observation locations:")
    print(f"  RMSE: {all_metrics['RMSE (m)']:.2f} m")
    print(f"  Bias: {all_metrics['Bias (m)']:.2f} m")
    
    if all_metrics['RMSE (m)'] < 1.0:
        print("\n  → Model fits observations well at these locations")
    elif all_metrics['RMSE (m)'] < 2.0:
        print("\n  → Model fit is acceptable at these locations")
    else:
        print("\n  → Model fit is poor - consider recalibration")
    
    if abs(all_metrics['Bias (m)']) < 0.5:
        print("  → No systematic bias detected")
    elif all_metrics['Bias (m)'] > 0:
        print("  → Model tends to overpredict heads")
    else:
        print("  → Model tends to underpredict heads")
    
    print("\n" + "-"*70)
    print("REMINDER: This tells us about fit at 3 points in one transect.")
    print("We have NO information about model accuracy elsewhere.")

In [ ]:
# Scatter plot: Simulated vs Observed (all wells)
fig, ax = plt.subplots(figsize=(8, 8))

# Get axis limits
all_obs = all_results['observed'].dropna()
all_sim = all_results['simulated'].dropna()
min_val = min(all_obs.min(), all_sim.min()) - 1
max_val = max(all_obs.max(), all_sim.max()) + 1

# Plot all wells
ax.scatter(all_results['observed'], all_results['simulated'],
           c='red', s=150, alpha=0.7, edgecolors='black')
ax.plot([min_val, max_val], [min_val, max_val], 'k--', label='1:1 line')
ax.set_xlim(min_val, max_val)
ax.set_ylim(min_val, max_val)
ax.set_xlabel('Observed Head (m a.s.l.)')
ax.set_ylabel('Simulated Head (m a.s.l.)')

if all_metrics:
    ax.set_title(f'Figure 3: Simulated vs Observed Heads (n={all_metrics["N"]})\n'
                 f'RMSE={all_metrics["RMSE (m)"]:.2f} m, Bias={all_metrics["Bias (m)"]:.2f} m\n'
                 f'(Limited data - metrics have low statistical significance)')
ax.set_aspect('equal')
ax.grid(alpha=0.3)
ax.legend()

# Add well labels
for _, well in all_results.iterrows():
    ax.annotate(well['well_id'], (well['observed'], well['simulated']),
                xytext=(5, 5), textcoords='offset points', fontsize=10)

plt.tight_layout()
plt.show()

## 5 - Validation Against Independent Data Types

Another powerful validation approach is testing the model against data types that were **not used in calibration**. For groundwater models, this often includes:

- **Hydraulic gradients** (calculated from head differences)
- **Groundwater-surface water exchange** (compared to baseflow estimates)
- **Flow directions** (compared to conceptual model)

### 5.1 - Hydraulic Gradient Validation

We can check if simulated hydraulic gradients match expected values from the conceptual model or field observations.

In [ ]:
# Calculate hydraulic gradients from simulated heads
if 'head' in dir():
    # Get cell sizes
    delr = mf.dis.delr.array  # Column widths
    delc = mf.dis.delc.array  # Row heights
    
    # Calculate gradients (central differences)
    head_2d = head[0].copy()
    head_2d[head_2d < -999] = np.nan
    
    # Gradient in x-direction (dh/dx)
    grad_x = np.zeros_like(head_2d)
    grad_x[:, 1:-1] = (head_2d[:, 2:] - head_2d[:, :-2]) / (2 * np.mean(delr))
    
    # Gradient in y-direction (dh/dy) - note: row index increases southward
    grad_y = np.zeros_like(head_2d)
    grad_y[1:-1, :] = -(head_2d[2:, :] - head_2d[:-2, :]) / (2 * np.mean(delc))
    
    # Gradient magnitude
    grad_mag = np.sqrt(grad_x**2 + grad_y**2)
    grad_mag = np.ma.masked_where(np.isnan(head_2d), grad_mag)
    
    print("Hydraulic Gradient Statistics:")
    print(f"  Mean gradient magnitude: {np.nanmean(grad_mag):.4f} m/m")
    print(f"  Max gradient magnitude:  {np.nanmax(grad_mag):.4f} m/m")
    print(f"  Typical for alluvial aquifers: 0.001 - 0.01 m/m")

In [ ]:
# Plot hydraulic gradient magnitude
if 'grad_mag' in dir():
    fig, ax = plt.subplots(figsize=(12, 10))
    
    pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
    grad_plot = pmv.plot_array(grad_mag, cmap='YlOrRd', alpha=0.8)
    plt.colorbar(grad_plot, ax=ax, label='Hydraulic Gradient (m/m)', shrink=0.5)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('Figure 4: Hydraulic Gradient Magnitude')
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()

### 5.2 - River-Aquifer Exchange Validation

For models with river boundaries, the simulated groundwater-surface water exchange can be compared to:
- Baseflow estimates from stream gauging
- Literature values for similar rivers

In [ ]:
# Check river leakage from water balance
if 'water_balance' in dir():
    print("="*70)
    print("RIVER-AQUIFER EXCHANGE")
    print("="*70)
    
    # Look for river-related terms in water balance
    riv_row = water_balance[water_balance['Component'].str.contains('RIVER', case=False)]
    
    if len(riv_row) > 0:
        riv_in = riv_row['Inflow'].values[0]
        riv_out = riv_row['Outflow'].values[0]
        net_exchange = riv_in - riv_out
        
        print(f"\nRiver Package Exchange:")
        print(f"  River → Aquifer (gaining): {riv_in:,.1f} m³/day")
        print(f"  Aquifer → River (losing):  {riv_out:,.1f} m³/day")
        print(f"  Net exchange:              {net_exchange:,.1f} m³/day")
        
        if net_exchange > 0:
            print(f"\n  → River is predominantly GAINING groundwater")
        else:
            print(f"\n  → River is predominantly LOSING water to aquifer")
        
        # Typical comparison (these would come from field measurements)
        print(f"\n  Note: Compare with baseflow separation analysis if available")
    else:
        print("\nNo river package found in water balance.")

### 5.3 - Flow Direction Validation

Check that simulated flow directions match our conceptual understanding.

In [ ]:
# Plot flow vectors
if 'head' in dir():
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Plot heads as background
    head_masked = np.ma.masked_where(head[0] < -999, head[0])
    pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
    head_plot = pmv.plot_array(head_masked, cmap='viridis', alpha=0.5)
    plt.colorbar(head_plot, ax=ax, label='Head (m a.s.l.)', shrink=0.5)
    
    # Add contours
    contours = pmv.contour_array(head_masked, levels=15, colors='black', linewidths=0.5)
    ax.clabel(contours, inline=True, fontsize=8, fmt='%.0f')
    
    # Calculate and plot flow vectors (subsampled for clarity)
    if 'grad_x' in dir() and 'grad_y' in dir():
        # Get cell centers
        xc = mf.modelgrid.xcellcenters
        yc = mf.modelgrid.ycellcenters
        
        # Subsample for plotting
        skip = max(1, min(mf.nrow, mf.ncol) // 20)
        
        # Flow is opposite to gradient
        qx = -grad_x
        qy = -grad_y
        
        ax.quiver(xc[::skip, ::skip], yc[::skip, ::skip],
                  qx[::skip, ::skip], qy[::skip, ::skip],
                  color='white', alpha=0.8, scale=0.1, width=0.003)
    
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('Figure 5: Simulated Head Distribution and Flow Directions')
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    print("\nFlow Direction Check:")
    print("  Compare flow arrows with expected flow direction from conceptual model.")
    print("  - Flow should generally follow topographic gradient")
    print("  - Flow should converge toward rivers (if gaining)")
    print("  - Flow should diverge from recharge areas")

## 6 - Physical Plausibility Assessment

Even if metrics look good, we must verify that results are physically reasonable.

In [ ]:
print("="*70)
print("PHYSICAL PLAUSIBILITY ASSESSMENT")
print("="*70)

# 1. Head bounds check
print("\n1. Simulated Head Bounds:")
head_valid = head[head > -999]
print(f"   Minimum head: {head_valid.min():.2f} m a.s.l.")
print(f"   Maximum head: {head_valid.max():.2f} m a.s.l.")
print(f"   Head range:   {head_valid.max() - head_valid.min():.2f} m")

# Check if heads are within reasonable bounds for the Limmat Valley
# (approximately 380-450 m a.s.l. based on topography)
if head_valid.min() > 350 and head_valid.max() < 500:
    print("   ✓ Heads are within plausible range for Limmat Valley")
else:
    print("   ⚠ Check if head values are reasonable for this location")

# 2. Gradient check
print("\n2. Hydraulic Gradient:")
if 'grad_mag' in dir():
    mean_grad = np.nanmean(grad_mag)
    print(f"   Mean gradient: {mean_grad:.4f} m/m ({mean_grad*1000:.2f} m/km)")
    
    if 0.0001 < mean_grad < 0.05:
        print("   ✓ Gradient is within typical range for alluvial aquifers")
    else:
        print("   ⚠ Gradient may be unusually high or low")

# 3. Water balance reasonableness
print("\n3. Water Balance Components:")
if 'water_balance' in dir():
    total_flow = water_balance['Inflow'].sum()
    
    # Check recharge
    rch_row = water_balance[water_balance['Component'].str.contains('RECHARGE', case=False)]
    if len(rch_row) > 0:
        rch_in = rch_row['Inflow'].values[0]
        rch_pct = rch_in / total_flow * 100
        print(f"   Recharge: {rch_in:,.0f} m³/day ({rch_pct:.1f}% of total inflow)")
    
    print(f"   Total throughflow: {total_flow:,.0f} m³/day")

In [ ]:
# 4. Parameter plausibility
print("\n4. Calibrated Parameter Values:")

# Check hydraulic conductivity
if hasattr(mf, 'upw'):
    hk = mf.upw.hk.array
elif hasattr(mf, 'lpf'):
    hk = mf.lpf.hk.array
else:
    hk = None

if hk is not None:
    hk_valid = hk[hk > 0]
    print(f"   Hydraulic Conductivity (K):")
    print(f"     Min: {hk_valid.min():.2e} m/day")
    print(f"     Max: {hk_valid.max():.2e} m/day")
    print(f"     Mean: {hk_valid.mean():.2e} m/day")
    
    # Typical K for gravel aquifers: 10-1000 m/day
    if 1 < hk_valid.mean() < 1000:
        print("   ✓ K values are reasonable for gravel/sand aquifer")
    else:
        print("   ⚠ Check K values against literature/field data")

# Check recharge
if hasattr(mf, 'rch') and mf.rch is not None:
    rch_array = mf.rch.rech.array
    if rch_array.ndim == 4:
        rch_2d = rch_array[0, 0]
    elif rch_array.ndim == 3:
        rch_2d = rch_array[0]
    else:
        rch_2d = rch_array
    
    rch_valid = rch_2d[rch_2d > 0]
    if len(rch_valid) > 0:
        rch_mm_yr = np.mean(rch_valid) * 1000 * 365
        print(f"\n   Recharge Rate:")
        print(f"     Mean: {rch_mm_yr:.1f} mm/year")
        
        # Typical recharge for temperate climates: 100-400 mm/year
        if 50 < rch_mm_yr < 600:
            print("   ✓ Recharge rate is reasonable for temperate climate")
        else:
            print("   ⚠ Check recharge rate against precipitation data")

## 7 - Residual Analysis

Examine spatial patterns in residuals to identify systematic model errors.

In [ ]:
# Spatial residual map
fig, ax = plt.subplots(figsize=(12, 10))

# Plot head distribution as background
head_masked = np.ma.masked_where(head[0] < -999, head[0])
pmv = flopy.plot.PlotMapView(model=mf, ax=ax)
head_plot = pmv.plot_array(head_masked, cmap='viridis', alpha=0.5)

# Plot all wells colored by residual
scatter = ax.scatter(
    all_results['x'],
    all_results['y'],
    c=all_results['residual'],
    cmap='RdBu_r',
    s=300,
    edgecolors='black',
    linewidth=2,
    vmin=-3, vmax=3,
    zorder=5
)
cbar = plt.colorbar(scatter, ax=ax, label='Residual (Sim - Obs) [m]', shrink=0.5)

# Add well labels with residual values
for _, well in all_results.iterrows():
    ax.annotate(f"{well['well_id']}\n({well['residual']:+.1f} m)",
                (well['x'], well['y']),
                xytext=(10, 10), textcoords='offset points', fontsize=9,
                fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Figure 6: Spatial Distribution of Residuals\n(Blue = underprediction, Red = overprediction)')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\nNote: With only 3 wells in a single transect, we cannot identify")
print("spatial patterns or systematic biases across the model domain.")

In [ ]:
# Simple residual summary (histogram not meaningful with n=3)
print("="*70)
print("RESIDUAL SUMMARY")
print("="*70)

all_residuals = all_results['residual'].dropna()

print(f"\nResiduals at observation wells:")
for _, well in all_results.iterrows():
    status = "overpredict" if well['residual'] > 0 else "underpredict"
    print(f"  {well['well_id']}: {well['residual']:+.2f} m ({status})")

print(f"\nSummary statistics:")
print(f"  Mean residual (bias): {all_residuals.mean():+.2f} m")
print(f"  Residual range: {all_residuals.min():.2f} to {all_residuals.max():.2f} m")
print(f"  Std deviation: {all_residuals.std():.2f} m")

# Note about limitations
print("\n" + "-"*70)
print("Note: A histogram or box plot is not meaningful with only 3 data points.")
print("With more observation wells, we would examine the residual distribution")
print("to check for normality and identify outliers.")

In [ ]:
# Residual pattern analysis - limited by data
print("="*70)
print("RESIDUAL PATTERN ANALYSIS")
print("="*70)

# Check for bias
mean_residual = all_results['residual'].mean()
print(f"\nMean residual: {mean_residual:.3f} m")
if abs(mean_residual) < 0.5:
    print("→ No significant systematic bias detected at observed locations")
else:
    if mean_residual > 0:
        print("→ Model tends to OVERPREDICT heads at observed locations")
    else:
        print("→ Model tends to UNDERPREDICT heads at observed locations")

print("\n" + "-"*70)
print("WHAT WE CANNOT ASSESS (due to limited data):")
print("-"*70)
print("""
With only 3 wells in a single transect, we cannot:

  ✗ Detect systematic spatial patterns across the domain
  ✗ Assess if errors correlate with distance from boundaries
  ✗ Identify zones where model performs poorly
  ✗ Evaluate if errors relate to geological features
  ✗ Determine if river conductance is appropriate
  ✗ Validate model behavior near pumping wells
  
These assessments would require observation wells distributed across
the model domain, particularly:
  - Near model boundaries
  - At varying distances from the river
  - In areas of different geological units
  - Near stress features (wells, drains, etc.)
""")

## 8 - Validation Summary

Compile all validation results into a summary assessment.

In [ ]:
print("="*70)
print("VALIDATION SUMMARY")
print("="*70)

# Create summary table - focusing on what we CAN assess
summary_data = []

# 1. Mass balance (verification - doesn't need observation data)
if 'discrepancy' in dir():
    status = "✓ PASS" if abs(discrepancy) < 0.1 else "⚠ CHECK"
    summary_data.append({
        'Check': 'Mass Balance (Verification)',
        'Criterion': '< 0.1%',
        'Result': f'{abs(discrepancy):.4f}%',
        'Status': status,
        'Data Needed': 'None (internal check)'
    })

# 2. Model fit at observations
if all_metrics:
    status = "✓ PASS" if all_metrics['RMSE (m)'] < 2.0 else "⚠ CHECK"
    summary_data.append({
        'Check': 'Fit at Observations',
        'Criterion': 'RMSE < 2.0 m',
        'Result': f"{all_metrics['RMSE (m)']:.2f} m",
        'Status': status,
        'Data Needed': f"n={all_metrics['N']} (INSUFFICIENT)"
    })

# 3. Bias
if all_metrics:
    status = "✓ PASS" if abs(all_metrics['Bias (m)']) < 0.5 else "⚠ CHECK"
    summary_data.append({
        'Check': 'Systematic Bias',
        'Criterion': '< ±0.5 m',
        'Result': f"{all_metrics['Bias (m)']:.3f} m",
        'Status': status,
        'Data Needed': f"n={all_metrics['N']} (INSUFFICIENT)"
    })

# 4. Independent validation - CANNOT ASSESS
summary_data.append({
    'Check': 'Independent Validation',
    'Criterion': 'Val/Cal RMSE < 1.5',
    'Result': 'N/A',
    'Status': '✗ CANNOT ASSESS',
    'Data Needed': 'Need n≥10 wells'
})

# 5. Spatial coverage - CANNOT ASSESS
summary_data.append({
    'Check': 'Spatial Coverage',
    'Criterion': 'Wells across domain',
    'Result': 'Single transect only',
    'Status': '✗ INSUFFICIENT',
    'Data Needed': 'Wells in multiple areas'
})

summary_df = pd.DataFrame(summary_data)
display(summary_df)

# Overall assessment
print("\n" + "="*70)
print("OVERALL VALIDATION ASSESSMENT")
print("="*70)

print("""
VERIFICATION (Mathematical Correctness):
  ✓ Mass balance is acceptable - model solves equations correctly
  
MODEL FIT (at observed locations):
  The model fits the 3 available observation wells reasonably well.
  However, this tells us nothing about accuracy elsewhere in the domain.

VALIDATION (Predictive Capability):
  ✗ CANNOT BE PROPERLY ASSESSED with available data
  
  With only 3 observation wells in a single transect:
  - No independent validation dataset possible
  - No spatial coverage to assess domain-wide accuracy
  - Statistical metrics have no significance
  
MODEL STATUS: UNVALIDATED
  
  This model should be considered a SCREENING TOOL only until
  additional observation data enables proper validation.
""")

## 9 - Discussion: Data Requirements and Professional Communication

### The Reality of Data Limitations

This case study illustrates a common challenge in groundwater modelling: **insufficient observation data for rigorous validation**. With only 3 observation wells in a single transect, we simply cannot assess whether this model makes reliable predictions across its domain.

### Communicating with Clients

As a professional modeller, you have an ethical obligation to communicate data limitations honestly. When faced with insufficient validation data, you need to:

1. **Clearly explain the limitation** - What data is missing and why it matters
2. **Describe the implications** - What the model can and cannot be used for
3. **Recommend solutions** - What additional data would enable proper validation
4. **Provide cost/benefit context** - Help clients make informed decisions about data collection

This is not about being negative - it's about building trust through transparency and helping clients make informed decisions.

### What Proper Validation Would Require

For a model of this size and complexity, proper validation would typically need:

| Requirement | Minimum | Recommended | Our Data |
|-------------|---------|-------------|----------|
| Observation wells | 10 | 20-30 | **3** |
| Spatial coverage | Multiple zones | Full domain | **Single transect** |
| Cal/Val split | 70/30 | 75/25 | **Not possible** |
| Independent data types | 1 | 2-3 | **Limited** |

### Model Use Categories Based on Validation Status

| Validation Status | Appropriate Uses | Inappropriate Uses |
|-------------------|------------------|--------------------|
| **Unvalidated** (our case) | Screening, conceptual understanding, identifying data gaps | Regulatory decisions, quantitative predictions, design |
| **Partially validated** | Preliminary design, relative comparisons | Final design, permit applications |
| **Fully validated** | Regulatory submissions, detailed predictions, design optimization | - |

### Key Lessons

1. **Be honest about limitations** - It's better to acknowledge what you don't know than to overstate confidence
2. **Quantify uncertainty where possible** - Even rough bounds are better than no bounds
3. **Recommend solutions** - Don't just identify problems; suggest how to address them
4. **Document everything** - Future modellers and decision-makers need to understand the model's limitations
5. **Validation is not optional** - A model without validation is a hypothesis, not a predictive tool

### When is a Model "Good Enough"?

This depends entirely on the **intended use**:

- **Conceptual understanding**: Our model is adequate - it provides insights into the flow system
- **Screening analysis**: Our model is adequate - it can identify potential issues for further study
- **Regulatory decisions**: Our model is **NOT** adequate - needs proper validation
- **Design of remediation systems**: Our model is **NOT** adequate - predictions carry unknown uncertainty

## Next Steps

Despite the validation limitations, we can still proceed with further analysis, keeping the model's unvalidated status in mind:

1. **Sensitivity Analysis** (Notebook 7): Identify which parameters most influence predictions - this helps prioritize what data to collect
2. **Uncertainty Quantification**: Even without validation, we can explore parameter uncertainty
3. **Scenario Modelling**: Use for screening-level "what-if" analysis, with appropriate caveats

### If Additional Data Becomes Available

With more observation wells, return to this notebook and:
1. Re-run the data adequacy assessment
2. Split data into calibration and validation sets
3. Perform proper independent validation
4. Update the model status accordingly

---

## References

### Validation Guidance

- Anderson, M.P., Woessner, W.W., & Hunt, R.J. (2015). *Applied Groundwater Modeling: Simulation of Flow and Advective Transport* (2nd ed.). Academic Press. [ScienceDirect](https://www.sciencedirect.com/book/9780120581030/applied-groundwater-modeling)

- Hill, M.C., & Tiedeman, C.R. (2007). *Effective Groundwater Model Calibration: With Analysis of Data, Sensitivities, Predictions, and Uncertainty*. Wiley. [Wiley Online Library](https://onlinelibrary.wiley.com/doi/book/10.1002/0470041080)

- Refsgaard, J.C., & Henriksen, H.J. (2004). Modelling guidelines - terminology and guiding principles. *Advances in Water Resources*, 27(1), 71-82. [ScienceDirect](https://www.sciencedirect.com/science/article/abs/pii/S0309170803001489)

### Best Practices

- Barnett, B., et al. (2012). *Australian Groundwater Modelling Guidelines*. Waterlines Report No. 82, National Water Commission. [PDF Download](https://www.epa.govt.nz/assets/FileAPI/proposal/NSP000028/Hearings-Week-01/c2cd52352e/02-Australian-Groundwater-modelling-guidelines.pdf)

- Reilly, T.E., & Harbaugh, A.W. (2004). Guidelines for evaluating ground-water flow models. *USGS Scientific Investigations Report 2004-5038*. [USGS Publications](https://pubs.usgs.gov/sir/2004/5038/)